In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [2]:
d_out = 2  # desired output dimension
d_in = inputs.shape[1] # input dimension

In [3]:
w_query = torch.nn.Parameter(torch.randn(d_in, d_out))  # W^Q
w_key   = torch.nn.Parameter(torch.randn(d_in, d_out))  # W^K
w_value = torch.nn.Parameter(torch.randn(d_in, d_out))  # W^V

In [4]:
print("w_query = ", w_query)
print("w_key = ", w_key)
print("w_value = ", w_value)

w_query =  Parameter containing:
tensor([[-0.0038,  0.0844],
        [ 0.2761, -0.0644],
        [ 0.1660,  2.4714]], requires_grad=True)
w_key =  Parameter containing:
tensor([[-0.4068, -0.4071],
        [ 0.4004, -0.2234],
        [-1.4173, -0.3151]], requires_grad=True)
w_value =  Parameter containing:
tensor([[ 1.8861, -1.1463],
        [-0.5135, -0.8996],
        [ 0.2423,  0.6856]], requires_grad=True)


In [5]:
query = inputs @ w_query  # Q = X * W^Q
key   = inputs @ w_key    # K = X * W^K
value = inputs @ w_value  # V = X * W^V
print("query = ", query)
print("key = ", key)

query =  tensor([[0.1875, 2.2262],
        [0.3476, 1.6215],
        [0.3387, 1.5750],
        [0.2141, 0.7967],
        [0.0827, 0.2960],
        [0.3120, 1.3119]], grad_fn=<MmBackward0>)
key =  tensor([[-1.3763, -0.4891],
        [-0.8108, -0.6263],
        [-0.7986, -0.6237],
        [-0.3250, -0.3232],
        [-0.3549, -0.4009],
        [-0.4795, -0.3724]], grad_fn=<MmBackward0>)


In [6]:
atten_score = query @ key.T  # Attention Score = Q * K^T
print("atten_score = ", atten_score)
atten_weight = torch.softmax(atten_score, dim=1)  # Attention Weight = softmax(Attention Score)
print("atten_weight = ", atten_weight)

atten_score =  tensor([[-1.3467, -1.5462, -1.5381, -0.7803, -0.9589, -0.9190],
        [-1.2714, -1.2974, -1.2889, -0.6369, -0.7734, -0.7706],
        [-1.2364, -1.2610, -1.2528, -0.6190, -0.7516, -0.7490],
        [-0.6843, -0.6726, -0.6678, -0.3270, -0.3954, -0.3994],
        [-0.2585, -0.2524, -0.2506, -0.1225, -0.1480, -0.1499],
        [-1.0709, -1.0746, -1.0673, -0.5253, -0.6366, -0.6382]],
       grad_fn=<MmBackward0>)
atten_weight =  tensor([[0.1349, 0.1105, 0.1114, 0.2376, 0.1988, 0.2069],
        [0.1229, 0.1197, 0.1207, 0.2317, 0.2022, 0.2027],
        [0.1240, 0.1210, 0.1220, 0.2299, 0.2013, 0.2019],
        [0.1404, 0.1421, 0.1427, 0.2007, 0.1874, 0.1867],
        [0.1565, 0.1574, 0.1577, 0.1793, 0.1747, 0.1744],
        [0.1280, 0.1276, 0.1285, 0.2209, 0.1977, 0.1973]],
       grad_fn=<SoftmaxBackward0>)


In [9]:
context_length = inputs.shape[0]
one_matrix = torch.ones(context_length, context_length)
mask_matrix = torch.tril(one_matrix)
print(mask_matrix)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [11]:
masked_atten_weight = torch.tril(atten_weight) # or atten_weight * mask_matrix 
print( masked_atten_weight)

tensor([[0.1349, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1229, 0.1197, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1240, 0.1210, 0.1220, 0.0000, 0.0000, 0.0000],
        [0.1404, 0.1421, 0.1427, 0.2007, 0.0000, 0.0000],
        [0.1565, 0.1574, 0.1577, 0.1793, 0.1747, 0.0000],
        [0.1280, 0.1276, 0.1285, 0.2209, 0.1977, 0.1973]],
       grad_fn=<TrilBackward0>)


## Renormalizing The Matrix 

In [14]:
row_sums = masked_atten_weight.sum(dim=1, keepdim=True)
normalized_atten_weight = masked_atten_weight / row_sums
print(normalized_atten_weight)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5065, 0.4935, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3379, 0.3297, 0.3324, 0.0000, 0.0000, 0.0000],
        [0.2243, 0.2270, 0.2280, 0.3207, 0.0000, 0.0000],
        [0.1895, 0.1907, 0.1910, 0.2171, 0.2117, 0.0000],
        [0.1280, 0.1276, 0.1285, 0.2209, 0.1977, 0.1973]],
       grad_fn=<DivBackward0>)


<div class="alert alert-block alert-success">

The softmax function converts its inputs into a probability distribution. 

When negative
infinity values (-∞) are present in a row, the softmax function treats them as zero
probability. 

(Mathematically, this is because e
-∞ approaches 0.)


We can implement this more efficient masking "trick" by creating a mask with 1's above
the diagonal and then replacing these 1's with negative infinity (-inf) values:

</div>

In [22]:
masked_upper = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(masked_upper)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [23]:
masked_matrix =atten_score.masked_fill(masked_upper.bool() , -torch.inf)
print(masked_matrix)

tensor([[-1.3467,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-1.2714, -1.2974,    -inf,    -inf,    -inf,    -inf],
        [-1.2364, -1.2610, -1.2528,    -inf,    -inf,    -inf],
        [-0.6843, -0.6726, -0.6678, -0.3270,    -inf,    -inf],
        [-0.2585, -0.2524, -0.2506, -0.1225, -0.1480,    -inf],
        [-1.0709, -1.0746, -1.0673, -0.5253, -0.6366, -0.6382]],
       grad_fn=<MaskedFillBackward0>)


In [25]:
causal_atten_weight = torch.softmax(masked_matrix,dim =1 )
print(causal_atten_weight)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5065, 0.4935, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3379, 0.3297, 0.3324, 0.0000, 0.0000, 0.0000],
        [0.2243, 0.2270, 0.2280, 0.3207, 0.0000, 0.0000],
        [0.1895, 0.1907, 0.1910, 0.2171, 0.2117, 0.0000],
        [0.1280, 0.1276, 0.1285, 0.2209, 0.1977, 0.1973]],
       grad_fn=<SoftmaxBackward0>)


## Dropout

In [27]:
example = torch.ones(6, 6) #B
print(example)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])


In [28]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) #A
example = torch.ones(6, 6) #B 
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


<div class="alert alert-block alert-info">

When applying dropout to an attention weight matrix with a rate of 50%, half of the
elements in the matrix are randomly set to zero. 

To compensate for the reduction in active
elements, the values of the remaining elements in the matrix are scaled up by a factor of
1/0.5 =2. 

This scaling is crucial to maintain the overall balance of the attention weights,
ensuring that the average influence of the attention mechanism remains consistent during
both the training and inference phases.
</div>

In [29]:
print(dropout(atten_weight))

tensor([[0.2697, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2415, 0.0000, 0.4044, 0.4055],
        [0.0000, 0.2419, 0.2439, 0.4597, 0.4027, 0.4037],
        [0.0000, 0.0000, 0.2855, 0.4014, 0.0000, 0.3734],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3495, 0.3488],
        [0.2560, 0.2551, 0.0000, 0.4419, 0.3953, 0.3947]],
       grad_fn=<MulBackward0>)


## IMPLEMENTING A COMPACT CAUSAL ATTENTION CLASS

<div class="alert alert-block alert-success">

In this section, we will now incorporate the causal attention and dropout modifications into
the SelfAttention Python class we developed in section 3.4. 

This class will then serve as a
template for developing multi-head attention in the upcoming section.

</div>

<div class="alert alert-block alert-success">

Before we begin, one more thing is to ensure that the code can handle batches
consisting of more than one input. 

This will ensure that the CausalAttention class supports the batch
outputs produced by the data loader we implemented earlier.

</div>

<div class="alert alert-block alert-success">

For simplicity, to simulate such batch inputs, we duplicate the input text example:

</div>

<div class="alert alert-block alert-info">

 2 inputs with 6 tokens each, and each token has embedding dimension 3
    
</div>

In [31]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

torch.Size([2, 6, 3])


<div class="alert alert-block alert-info">

Step 1: Compared to the previous SelfAttention_v1 class, we added a dropout layer.
    
Step 2: The register_buffer call is also a new addition (more information is provided in the following text).

Step 3:  We transpose dimensions 1 and 2, keeping the batch dimension at the first position (0).

Step 4: In PyTorch, operations with a trailing underscore are performed in-place, avoiding unnecessary memory
copies
    
</div>